# Projeto Final — Álgebra Linear e Otimização para Machine Learning
## Notebook Guia e Validação Numérica em NumPy Puro

**Disciplina:** *Álgebra Linear e Otimização para Aprendizado de Máquina* \
**Professor:** *Marcos Medeiros Raimundo* \
**Instituição:** *Instituto de Computação — UNICAMP* \
**Semestre:** *2º Semestre de 2026* \

[← Voltar ao Enunciado Oficial do Projeto](https://ic.unicamp.br/mraimundo/teaching/optimization-linear-algebra/projeto.html)

| Nome do Integrante | RA |
|---|---|
| *Integrante 1* | *RA 1* |
| *Integrante 2* | *RA 2* |
| *Integrante 3 (opcional)* | *RA 3* |

> **Regra de Composição:** Equipes de **2 ou 3 alunos, impreterivelmente**. Não são aceitos trabalhos individuais nem grupos com 4 ou mais integrantes.

---

### Diretrizes Metodológicas Mandatórias

1. **NumPy Puro Exclusivo:** Todas as rotinas, métricas, decomposições matriciais, sistemas lineares e otimizadores devem ser implementados **exclusivamente com `numpy`** (e `matplotlib` para visualizações). É expressamente proibido o uso de chamadas de solvers automáticos de alto nível (`scikit-learn`, `scipy.optimize`, `pytorch`, etc.).
2. **Memorial Passo a Passo:** Cada verificação numérica deste notebook deve corresponder à sua derivação analítica formal apresentada no Relatório Técnico em PDF.
3. **Tolerância Numérica:** Operações algébricas devem ser checadas respeitando tolerâncias de ponto flutuante ($10^{-10}$ a $10^{-15}$).

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Configuração de semente e precisão de exibição
np.random.seed(42)
np.set_printoptions(precision=4, suppress=True)
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
print("Ambiente configurado com sucesso! NumPy versão:", np.__version__)

---
## Conjunto de Dados do Projeto

Definição canônica dos sistemas de dados estabelecidos no [Enunciado Oficial](https://ic.unicamp.br/mraimundo/teaching/optimization-linear-algebra/projeto.html):

1. **Dataset $\mathcal{D}$ (Avaliação Imobiliária):** $N=6$ amostras com $d=2$ atributos (Área $x_1$ e Idade $x_2$), alvo contínuo de preço $y$ e rótulo de classe $c \in \{0, 1\}$. Ponto de consulta: $\mathbf{x}_{\text{novo}} = [6{,}0, 20{,}0]^T$.
2. **Matriz-Toy $\mathcal{A}$:** Sistema $3 \times 2$ de posto completo para projeções ortogonais e SVD compacta.
3. **Matriz-Toy $\mathcal{B}$:** Sistema $3 \times 3$ com colinearidade linear exata ($\mathbf{x}_3 = \mathbf{x}_1 + \mathbf{x}_2$) para estudo de curvatura nula e Hessiana singular.

In [ ]:
# 1. Dataset D: Amostras imobiliárias (N=6, d=2)
X_data = np.array([
    [5.0, 30.0],
    [7.0, 38.0],
    [9.0, 40.0],
    [3.5, 19.0],
    [8.0, 15.0],
    [2.0, 35.0]
], dtype=np.float64)

y_prices = np.array([2.5, 3.8, 4.2, 1.9, 4.5, 1.2], dtype=np.float64)
c_labels = np.array([0, 0, 1, 0, 1, 0], dtype=np.int64)
x_novo = np.array([6.0, 20.0], dtype=np.float64)

print("Shape X_data:", X_data.shape)
print("Shape y_prices:", y_prices.shape)
print("Shape c_labels:", c_labels.shape)

In [ ]:

# 2. Matriz-Toy A e vetor y_A (Questão 3)
A_mat = np.array([
    [1.0, 1.0],
    [0.0, 1.0],
    [1.0, 0.0]
], dtype=np.float64)
y_A = np.array([4.0, 1.0, 1.0], dtype=np.float64)

# 3. Matriz-Toy B e vetor y_B (Questão 4 - note x_3 = x_1 + x_2)
B_mat = np.array([
    [2.0, 1.0, 3.0],
    [0.0, 2.0, 2.0],
    [1.0, 0.0, 1.0]
], dtype=np.float64)
y_B = np.array([5.0, 2.0, 1.0], dtype=np.float64)

print("Shape A_mat:", A_mat.shape)
print("Shape B_mat:", B_mat.shape)
print("Shape y_B:", y_B.shape)

---
## Questão 1: Normas, Métricas, Escala e o Algoritmo $k$-NN

**Objetivo:** Implementar em NumPy puro as funções de cálculo de distância ($L_1, L_2, L_\infty$ e Mahalanobis com matriz de covariância amostral $\mathbf{S}$), padronização de atributos (Z-score e MinMax), e o classificador/regressor $k$-NN ($k=3$).

> **Item a preencher:** Calcule as distâncias de cada amostra ao ponto $\mathbf{x}_{\text{novo}} = [6{,}0, 20{,}0]^T$ com e sem padronização, e discuta a distorção induzida pela escala da variável 'Idade' ($x_2$).

In [ ]:
# Implementações das normas em NumPy puro
def dist_l1(u: np.ndarray, v: np.ndarray) -> float:
    return float(np.sum(np.abs(u - v)))

def dist_l2(u: np.ndarray, v: np.ndarray) -> float:
    return float(np.sqrt(np.sum((u - v) ** 2)))

def dist_linf(u: np.ndarray, v: np.ndarray) -> float:
    return float(np.max(np.abs(u - v)))

def dist_mahalanobis(u: np.ndarray, v: np.ndarray, S_inv: np.ndarray) -> float:
    diff = u - v
    return float(np.sqrt(diff @ S_inv @ diff))

# Funções de padronização
def zscore_standardize(X: np.ndarray) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    mu = np.mean(X, axis=0)
    sigma = np.std(X, axis=0, ddof=1) # desvio-padrão amostral
    return (X - mu) / sigma, mu, sigma

def minmax_scale(X: np.ndarray) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    x_min = np.min(X, axis=0)
    x_max = np.max(X, axis=0)
    return (X - x_min) / (x_max - x_min), x_min, x_max

# Matriz de covariância amostral para Mahalanobis
S_cov = np.cov(X_data, rowvar=False)
S_inv = np.linalg.inv(S_cov)

print("Matriz de Covariância S:\n", S_cov)
print("\nDistâncias L2 dos 6 pontos brutos até x_novo:")
for i, x_i in enumerate(X_data, 1):
    print(f"  Amostra {i} ({x_i}): d_L2 = {dist_l2(x_i, x_novo):.4f}")

---
## Questão 2: Regressão Linear, Posto e Mínimos Quadrados Clássicos (OLS)

**Objetivo:** Formular a matriz de projeto com termo independente $\mathbf{X} = [\mathbf{1}, \mathbf{x}_1, \mathbf{x}_2] \in \mathbb{R}^{6 \times 3}$, calcular a matriz de Gram $\mathbf{X}^T \mathbf{X}$, resolver as equações normais $(\mathbf{X}^T \mathbf{X})\hat{\mathbf{w}} = \mathbf{X}^T \mathbf{y}$ e verificar a ortogonalidade fundamental dos resíduos: $\mathbf{X}^T \mathbf{e} = \mathbf{0}$.

In [ ]:
# Montando matriz de projeto com coluna de uns (intercepto)
N = len(X_data)
X_design = np.column_stack([np.ones(N), X_data])

# Matriz de Gram
Gram = X_design.T @ X_design
print("Matriz de Gram X^T X:\n", Gram)
print("Posto de X:", np.linalg.matrix_rank(X_design))
print("Determinante de X^T X:", np.linalg.det(Gram))

# Solução OLS via Equações Normais: w_hat = (X^T X)^(-1) X^T y
w_hat = np.linalg.inv(Gram) @ (X_design.T @ y_prices)
print("\nVetor de Pesos OLS [w0 (bias), w1 (area), w2 (idade)]:", w_hat)

# Resíduos e Checagem de Ortogonalidade: X^T * e = 0
y_hat = X_design @ w_hat
residuos = y_prices - y_hat
ortogonalidade = X_design.T @ residuos

print("Norma dos resíduos ||e||_2:", np.linalg.norm(residuos))
print("Verificação X^T e (deve ser ~ 0):", ortogonalidade)
assert np.allclose(ortogonalidade, 0, atol=1e-10), "Erro: Resíduos não são ortogonais ao espaço de colunas!"
print("✅ Ortogonalidade comprovada com sucesso!")

# Predição para x_novo
x_novo_design = np.array([1.0, x_novo[0], x_novo[1]])
preco_predito = x_novo_design @ w_hat
print(f"\nPreço predito para x_novo {x_novo}: {preco_predito:.4f} centenas de milhares de reais.")

---
## Questão 3: Projeção Ortogonal, Teorema Espectral e SVD Compacta

**Objetivo:** Construir a matriz de projeção ortogonal $\mathbf{P} = \mathcal{A}(\mathcal{A}^T \mathcal{A})^{-1} \mathcal{A}^T$, testar suas propriedades (simetria $\mathbf{P} = \mathbf{P}^T$ e idempotência $\mathbf{P}^2 = \mathbf{P}$), deduzir analiticamente e verificar numericamente a SVD compacta $\mathcal{A} = \mathbf{U} \boldsymbol{\Sigma} \mathbf{V}^T$ e calcular a pseudoinversa de Moore-Penrose $\mathcal{A}^+$.

In [ ]:
# Matriz de Projeção Ortogonal no Col(A)
AtA = A_mat.T @ A_mat
P_proj = A_mat @ np.linalg.inv(AtA) @ A_mat.T

print("Matriz de Projeção P:\n", P_proj)
print("P é simétrica?", np.allclose(P_proj, P_proj.T, atol=1e-14))
print("P é idempotente (P^2 = P)?", np.allclose(P_proj @ P_proj, P_proj, atol=1e-14))
print("Traço de P (posto de P):", np.trace(P_proj))

# Decomposição Espectral de A^T A e SVD Compacta manual
eigvals, V = np.linalg.eigh(AtA)
# Ordenar autovalores decrescentes
idx = np.argsort(eigvals)[::-1]
eigvals, V = eigvals[idx], V[:, idx]
sigmas = np.sqrt(eigvals)

# Vetores u_i = (1 / sigma_i) * A * v_i
U = np.column_stack([(1.0 / sigmas[i]) * (A_mat @ V[:, i]) for i in range(len(sigmas))])
Sigma_mat = np.diag(sigmas)

# Reconstrução A_reconstructed = U * Sigma * V^T
A_rec = U @ Sigma_mat @ V.T
print("\nSVD Reconstrução correta?", np.allclose(A_rec, A_mat, atol=1e-14))

# Pseudoinversa de Moore-Penrose: A^+ = V * Sigma^(-1) * U^T
A_pinv_manual = V @ np.diag(1.0 / sigmas) @ U.T
A_pinv_numpy = np.linalg.pinv(A_mat)
print("A^+ manual coincide com np.linalg.pinv?", np.allclose(A_pinv_manual, A_pinv_numpy, atol=1e-14))
print("Solução de mínimos quadrados w_A = A^+ y_A:", A_pinv_manual @ y_A)

---
## Questão 4: Cálculo Diferenciável, Hessiana e Vales Planos (Ridge Regression)

**Objetivo:** Analisar a função de custo $f(\mathbf{w}) = \frac{1}{2} \|\mathcal{B}\mathbf{w} - \mathbf{y}_{\mathcal{B}}\|_2^2$ sobre a matriz $\mathcal{B}$ (com colinearidade linear $\mathbf{x}_3 = \mathbf{x}_1 + \mathbf{x}_2$).

Provar que $\det(\mathcal{B}^T \mathcal{B}) = 0$, identificar o autovalor nulo $\lambda_3 = 0$, explicitar o vale plano e demonstrar como a regularização de Ridge $(\mathcal{B}^T \mathcal{B} + \lambda \mathbf{I})$ restaura a estrita convexidade da Hessiana.

In [ ]:
# Hessiana de f(w): H = B^T * B
H_B = B_mat.T @ B_mat
print("Hessiana H = B^T B:\n", H_B)
print("Determinante de H:", np.linalg.det(H_B))

# Autovalores e Autovetores de H
eigvals_B, eigvecs_B = np.linalg.eigh(H_B)
print("Autovalores de B^T B:", eigvals_B)

# Vetor no núcleo (vale plano): B * [1, 1, -1]^T = 0
v_null = np.array([1.0, 1.0, -1.0])
print("B @ [1, 1, -1]^T (direção do vale plano):", B_mat @ v_null)

# Regularização Ridge: (B^T B + lambda * I)
lambda_reg = 1
H_ridge = H_B + lambda_reg * np.eye(3)
eigvals_ridge = np.linalg.eigvalsh(H_ridge)
print(f"\nAutovalores da Hessiana com Ridge (lambda={lambda_reg}):", eigvals_ridge)
print("Todos os autovalores são estritamente positivos?", np.all(eigvals_ridge > 0))

# Solução Ridge w_ridge
w_ridge = np.linalg.solve(H_ridge, B_mat.T @ y_B)
print("Solução Ridge w_ridge:", w_ridge)

---
## Questão 5: Análise de Convergência da Descida do Gradiente e Regressão Logística

**Objetivo:**
1. Calcular o número de condição $\kappa = \lambda_{\max}/\lambda_{\min}$ e a taxa ótima $\alpha^* = \frac{2}{\lambda_{\max} + \lambda_{\min}}$;
2. Implementar a Regressão Logística Binária (perda de entropia cruzada) no Dataset $\mathcal{D}$ com os rótulos de perfil $c_i \in \{0, 1\}$;
3. Comparar a convergência da Descida do Gradiente de 1ª ordem contra o **Método de Newton de 2ª ordem** (Newton-Raphson).

In [ ]:
# Função Sigmoide estável
def sigmoid(z: np.ndarray) -> np.ndarray:
    return np.where(z >= 0, 1.0 / (1.0 + np.exp(-z)), np.exp(z) / (1.0 + np.exp(z)))

# Perda Logística (Cross-Entropy)
def loss_logistic(w: np.ndarray, X: np.ndarray, y: np.ndarray) -> float:
    p = sigmoid(X @ w)
    eps = 1e-15
    p = np.clip(p, eps, 1.0 - eps)
    return float(-np.mean(y * np.log(p) + (1.0 - y) * np.log(1.0 - p)))

# Gradiente da Perda Logística: grad = (1/N) * X^T (p - y)
def grad_logistic(w: np.ndarray, X: np.ndarray, y: np.ndarray) -> np.ndarray:
    p = sigmoid(X @ w)
    return (X.T @ (p - y)) / len(y)

# Hessiana da Perda Logística: H = (1/N) * X^T S X, onde S = diag(p * (1 - p))
def hess_logistic(w: np.ndarray, X: np.ndarray) -> np.ndarray:
    p = sigmoid(X @ w)
    s = p * (1.0 - p)
    return (X.T * s) @ X / len(X)

# Padronizando X para regressão logística
X_norm, _, _ = zscore_standardize(X_data)
X_log_design = np.column_stack([np.ones(len(X_norm)), X_norm])

# 1. Otimização via Descida do Gradiente (GD)
w_gd = np.zeros(X_log_design.shape[1])
lr = 0.5
history_gd = []
for step in range(100):
    g = grad_logistic(w_gd, X_log_design, c_labels)
    history_gd.append(loss_logistic(w_gd, X_log_design, c_labels))
    w_gd -= lr * g
    if np.linalg.norm(g) < 1e-6:
        print(f"GD convergiu no passo {step}!")
        break

# 2. Otimização via Método de Newton (2ª Ordem)
w_newton = np.zeros(X_log_design.shape[1])
history_newton = []
for step in range(20):
    g = grad_logistic(w_newton, X_log_design, c_labels)
    H = hess_logistic(w_newton, X_log_design)
    history_newton.append(loss_logistic(w_newton, X_log_design, c_labels))
    # Atualização de Newton: w = w - H^(-1) * g
    w_newton -= np.linalg.solve(H + 1e-6 * np.eye(len(H)), g)
    if np.linalg.norm(g) < 1e-6:
        print(f"Método de Newton convergiu quadraticamente no passo {step}!")
        break

print("w_gd final:", w_gd)
print("w_newton final:", w_newton)

---
## Questão 6: Otimização Estocástica, Momento e Métodos Adaptativos

**Objetivo:** Implementar em NumPy puro os quatro algoritmos fundamentais de otimização de primeira ordem:
1. **GD Determinístico** (Full-Batch);
2. **SGD Estocástico** (com ruído amostral de mini-batch);
3. **GD com Momento de Polyak** (Heavy-Ball);
4. **Adam** (Adaptive Moment Estimation com estimativas de 1º e 2º momentos desviesadas).

Plotar a curva comparativa da perda ao longo das iterações.

In [ ]:
# Implementação dos otimizadores em NumPy puro
def run_optimizers(X: np.ndarray, y: np.ndarray, n_epochs: int = 60):
    d_in = X.shape[1]
    
    # 1. GD Puro
    w_gd = np.zeros(d_in)
    loss_gd = []
    for _ in range(n_epochs):
        loss_gd.append(loss_logistic(w_gd, X, y))
        w_gd -= 0.3 * grad_logistic(w_gd, X, y)
        
    # 2. GD com Momento (Polyak)
    w_mom = np.zeros(d_in)
    v_mom = np.zeros(d_in)
    beta = 0.9
    loss_mom = []
    for _ in range(n_epochs):
        loss_mom.append(loss_logistic(w_mom, X, y))
        g = grad_logistic(w_mom, X, y)
        v_mom = beta * v_mom + 0.3 * g
        w_mom -= v_mom
        
    # 3. Adam
    w_adam = np.zeros(d_in)
    m = np.zeros(d_in)
    v = np.zeros(d_in)
    beta1, beta2, eps = 0.9, 0.999, 1e-8
    lr = 0.1
    loss_adam = []
    for t in range(1, n_epochs + 1):
        loss_adam.append(loss_logistic(w_adam, X, y))
        g = grad_logistic(w_adam, X, y)
        m = beta1 * m + (1.0 - beta1) * g
        v = beta2 * v + (1.0 - beta2) * (g ** 2)
        m_hat = m / (1.0 - beta1 ** t)
        v_hat = v / (1.0 - beta2 ** t)
        w_adam -= lr * m_hat / (np.sqrt(v_hat) + eps)
        
    return loss_gd, loss_mom, loss_adam

loss_gd, loss_mom, loss_adam = run_optimizers(X_log_design, c_labels)

# Visualização Comparativa
plt.figure(figsize=(8, 4.5))
plt.plot(loss_gd, label='GD Padrão (Batch)', lw=2)
plt.plot(loss_mom, label='GD com Momento (Polyak, beta=0.9)', lw=2)
plt.plot(loss_adam, label='Adam Adaptativo', lw=2)
plt.xlabel('Época / Iteração')
plt.ylabel('Perda Logística (Cross-Entropy)')
plt.title('Comparativo de Otimizadores em NumPy Puro')
plt.legend()
plt.tight_layout()
plt.show()

---
## Questão Bônus: Otimização Não-Suave, Operadores Proximais e o Algoritmo ISTA (Lasso)

**Objetivo:** Demonstrar o operador de limiarização suave (*Soft-Thresholding*) e o algoritmo ISTA (*Iterative Shrinkage-Thresholding Algorithm*) para otimização da perda com regularização $L_1$:

$$\min_{\mathbf{w}} \frac{1}{2N} \|\mathbf{X}\mathbf{w} - \mathbf{y}\|_2^2 + \lambda \|\mathbf{w}\|_1$$

onde o operador proximal é dado analiticamente por:
$$\operatorname{prox}_{\alpha \lambda \|\cdot\|_1}(v)_j = \operatorname{sgn}(v_j) \max(|v_j| - \alpha \lambda, 0)$$

In [ ]:
def soft_thresholding(v: np.ndarray, thresh: float) -> np.ndarray:
    return np.sign(v) * np.maximum(np.abs(v) - thresh, 0.0)

def ista_lasso(X: np.ndarray, y: np.ndarray, lambda_l1: float = 0.2, n_iters: int = 100):
    N, d = X.shape
    # Maior autovalor de X^T X / N para garantir taxa segura alpha <= 1/L
    L = np.max(np.linalg.eigvalsh((X.T @ X) / N))
    alpha = 1.0 / L
    
    w = np.zeros(d)
    for _ in range(n_iters):
        # Gradiente do termo suave (erro quadrático)
        grad_smooth = (X.T @ (X @ w - y)) / N
        # Passo proximal com soft-thresholding
        w = soft_thresholding(w - alpha * grad_smooth, alpha * lambda_l1)
    return w

w_lasso = ista_lasso(X_design, y_prices, lambda_l1=0.3)
print("Pesos obtidos pelo ISTA (Lasso):")
print("w_lasso:", w_lasso)
print("Coeficientes exatamente nulos induzidos pela norma L1:", np.sum(w_lasso == 0))